# 문제 6

[Kaggle 형] train_prob.csv로 failure 예측하는 모델을 만들고, 

test_prob.csv에 대한 failure가 1일 확률 예측하여 다음과 같은 형식의 answer6.csv를 만들어라. 

측정 지표는 AUC(area under of ROC curve)이다. id 는 테스트 케이스의 id 이고, failure에는 failure가 1이 될 확률이다.

id,failure

16115, 0.1

16116, 0.2


**강사: 멀티캠퍼스 강선구(sunku0316.kang@multicampus.com, sun9sun9@gmail.com)**

In [1]:
# 실행 환경 확인

import pandas as pd
import numpy as np
import sklearn
import scipy
import statsmodels
import mlxtend # !pip install --upgrade mlxtend
import sys
import xgboost as xgb # !pip install --upgrade xgboost

print(sys.version)
for i in [pd, np, sklearn, scipy, mlxtend, statsmodels, xgb]:
    print(i.__name__, i.__version__)

3.7.4 (tags/v3.7.4:e09359112e, Jul  8 2019, 20:34:20) [MSC v.1916 64 bit (AMD64)]
pandas 0.25.1
numpy 1.18.5
sklearn 0.21.3
scipy 1.5.2
mlxtend 0.15.0.0
statsmodels 0.11.1
xgboost 0.80


# Kaggle형 풀이 단계

Step 0: Kaggle용 데이터셋을 만든다.

Step 1: 검증 방법을 정하고, 검증 루틴을 만듭니다.

Step 2: Baseline 모델을 만듭니다

Step 3: 모델 선택 루틴을 만듭니다.

|id|failure|
|----|----|
|16115| 0.1|
|16116| 0.2|

....	

Step 4: 모델 개선 작업을 합니다.

## Step 0: Kaggle용 데이터셋을 만든다.

In [5]:
# 데이터를 식별할 만한 변수(고윳값)가 있으면 인덱스로 사용합니다.
# 여기서는 id를 인덱스로 지정해볼만 합니다.
# 이 과정은 필수는 아닙니다.

df_train = pd.read_csv('train_prob.csv', index_col = ['id'])
df_test = pd.read_csv('test_prob.csv', index_col = ['id'])
s_kaggle_ans = pd.read_csv('test_prob_ans.csv', index_col = ['id'])['failure']

In [6]:
# from 문제 1
df_train = df_train.assign(
    na_1 = lambda x: x['measurement_3'].isna(),
    na_2 = lambda x: x['measurement_5'].isna(),
)
df_test = df_test.assign(
    na_1 = lambda x: x['measurement_3'].isna(),
    na_2 = lambda x: x['measurement_5'].isna(),
)

In [7]:
df_train['product_code'].value_counts()

C    5765
E    5343
B    5250
A    5100
Name: product_code, dtype: int64

In [8]:
df_test['product_code'].value_counts()

D    5112
Name: product_code, dtype: int64

In [10]:
# 방법 2: groupby ~ apply ~ fit_transform
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.linear_model import LinearRegression

X_imp = ['measurement_{}'.format(i) for i in range(3, 10)] + ['measurement_17']
imp = IterativeImputer(
    estimator = LinearRegression(fit_intercept = True), random_state=123
)

df_train[X_imp] = df_train.groupby('product_code')[X_imp].apply(
    lambda x: pd.DataFrame(imp.fit_transform(x), index = x.index, columns = X_imp)
)
df_test[X_imp] = df_test.groupby('product_code')[X_imp].apply(
    lambda x: pd.DataFrame(imp.fit_transform(x), index = x.index, columns = X_imp)
)

In [15]:
# 방법 2 : train / test를 통합하여 처리해봅니다.

from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

X_imp = ['measurement_{}'.format(i) for i in range(3, 10)] + ['measurement_17']
# train에 등장하지 않은 수준이 있습니다, test를 포함하여 결측처리 모델을 만듭니다.
s_imp = pd.concat([
        df_train[X_imp + ['product_code']],
        df_test[X_imp + ['product_code']]
], axis=0).groupby('product_code')\
.apply(
    lambda x: IterativeImputer(estimator=LinearRegression(),random_state=123).fit(x[X_imp])
)
# train에 적용합니다.
df_train[X_imp] = df_train[X_imp + ['product_code']]\
            .groupby('product_code')\
            .apply(
                lambda x: pd.DataFrame(s_imp.loc[x.name].transform(x[X_imp]), index=x.index, columns=X_imp)
            )
# test에 적용합니다.
df_test[X_imp] = df_test[X_imp + ['product_code']]\
            .groupby('product_code')\
            .apply(
                lambda x: pd.DataFrame(s_imp.loc[x.name].transform(x[X_imp]), index=x.index, columns=X_imp)
            )

In [17]:
# 방법 1: train / test 개별처리 합니다.
X_mean = ['measurement_{}'.format(i) for i in range(10, 17)]
df_train[X_mean] = df_train.groupby('product_code')[X_mean].transform(
    lambda x: x.fillna(x.mean())
)
df_test[X_mean] = df_test.groupby('product_code')[X_mean].transform(
    lambda x: x.fillna(x.mean())
)

In [18]:
# 방법 2: train / test를 통합하여 처리합니다.
X_mean = ['measurement_{}'.format(i) for i in range(10, 17)]
df_mean = pd.concat([
        df_train[X_mean + ['product_code']],
        df_test[X_mean + ['product_code']]
], axis=0).groupby('product_code').mean()

df_train[X_mean] = df_train[X_mean + ['product_code']].groupby('product_code')[X_mean].apply(
    lambda x: x.fillna(df_mean.loc[x.name])
).reset_index(level=0, drop=True)
df_test[X_mean] = df_test[X_mean + ['product_code']].groupby('product_code')[X_mean].apply(
    lambda x: x.fillna(df_mean.loc[x.name])
).reset_index(level=0, drop=True)

In [14]:
m = pd.concat(
    [df_train['loading'], df_test['loading']]
).mean()
df_train['loading'] = df_train['loading'].fillna(m)
df_test['loading'] = df_test['loading'].fillna(m)

In [57]:
df_train['loading_log'] = np.log(df_train['loading'])
df_test['loading_log'] = np.log(df_test['loading'])

문제1: na_1 = isna_3, na_2 = isna_5, failure와 연관

문제2: loading_log = loading -> log 변환 

       attribute_0, attribute_1 버리자

문제3: loading 평균 대체
      LR: STD  \['loading', 'measurement_1', 'measurement_4', 'measurement_14', 'measurement_17'\] pt: \[ 'na_1'\]

문제4: LDA predict, transform 
      
      LR: loading + STD ['measurement_0 ~ 17'] -> PCA(n_components = 7) best 
      
문제5: RandomForest {'n_estimators': 15, 'max_depth': 7, 'min_samples_split': 512}
      loading + 'measurement_0 ~ 17' + na_1, na_2

In [58]:
X_all = df_test.columns.tolist()
np.array(X_all)

array(['product_code', 'loading', 'attribute_0', 'attribute_1',
       'attribute_2', 'attribute_3', 'measurement_0', 'measurement_1',
       'measurement_2', 'measurement_3', 'measurement_4', 'measurement_5',
       'measurement_6', 'measurement_7', 'measurement_8', 'measurement_9',
       'measurement_10', 'measurement_11', 'measurement_12',
       'measurement_13', 'measurement_14', 'measurement_15',
       'measurement_16', 'measurement_17', 'na_1', 'na_2', 'loading_log'],
      dtype='<U14')

## Step1: 검증 방법을 정하고, 검증 루틴을 만듭니다.

In [59]:
from sklearn.model_selection import GroupKFold
gkf = GroupKFold(4)
for train_idx, valid_idx in gkf.split(df_train[X_all], df_train['failure'], groups = df_train['product_code']):
    print(df_train.iloc[train_idx]['product_code'].unique(), df_train.iloc[valid_idx]['product_code'].unique())

['A' 'B' 'E'] ['C']
['A' 'B' 'C'] ['E']
['A' 'C' 'E'] ['B']
['B' 'C' 'E'] ['A']


In [60]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import cross_validate
from sklearn.model_selection import GroupKFold
gkf = GroupKFold(4)

s_hist = list()
# 모델 검증 루틴입니다. (Step 2)
def eval_model(model_name, clf):
    """
        모델의 인스턴스를 받아 검증 결과(AUC) 를 구합니다.
        1. GroupKFold 검증 - df_train, groups - product_code
        2. df_train 대한 교차검증결과에 대한 AUC를 구합니다.
        3. 주어진 모델명으로 평가 결과를 저장합니다. Format: Valid: {:.5f}±{:.5f}, Train: {:.5f}±{:.5f}
        4. 가장 최근의 수행결과를 보여 주어 선택하는데 활용하도록 합니다.
    Parameters:
        model_name: str, 모델의 이름,
        clf: sklearn object, 모델 인스턴스
    """
    # 1. GroupKFold 검증 - df_train, groups - product_code
    # 2. df_train 대한 교차검증결과에 대한 AUC를 구합니다.
    result = cross_validate(
        clf, df_train[X_all], df_train['failure'], groups = df_train['product_code'], scoring = 'roc_auc', cv = gkf, 
        return_train_score = True
    )
    result_str = "Valid: {:.5f}±{:.5f}, Train: {:.5f}±{:.5f}".format(
        np.mean(result['test_score']), np.std(result['test_score']),
        np.mean(result['train_score']), np.std(result['train_score'])
    )
    s_hist.append(
        pd.Series([model_name, result_str], index = ['name', 'result'])
    )
    display(
        pd.DataFrame(s_hist).groupby('name').last()
    )
    
# 모델 선택 루틴입니다. (Step 3)
def select_model(clf):
    """
        1. 전체 학습데이터(df_train)로 학습을 합니다. 
        2. df_test에 대한 예측을 합니다.
        3. 예측 결과를 출력양식(id, failure)에 맞춰 csv파일을 만듭니다.
        4. 자가 채점을 위한 예측 결과를 반환합니다.
    Parameters: 
        clf: sklearn object, 모델 인스턴스
    Returns: 1차원 np.ndarray
        예측 결과
    """
    # 1. 전체 학습데이터(df_train)로 학습을 합니다. 
    clf.fit(df_train[X_all], df_train['failure'])
    # 2. df_test에 대한 예측을 합니다.
    prd = clf.predict_proba(df_test[X_all])[:, 1]
    # 3. 예측 결과를 출력양식(id, failure)에 맞춰 csv파일을 만듭니다.
    pd.DataFrame({
        'id': df_test.index,
        'failure': prd
    }).to_csv('answer6.csv', index = None)
    # 4. 자가 채점을 위한 예측 결과를 반환합니다.
    return prd

## Step2: Baseline 모델을 만듭니다.

std: \['loading', 'measurement_1', 'measurement_4', 'measurement_14', 'measurement_17'\] pt: \[ 'na_1'\] -> LR

In [61]:
from sklearn.linear_model import LogisticRegression
ct = ColumnTransformer([
    ('std', StandardScaler(), ['loading', 'measurement_1', 'measurement_4', 'measurement_14', 'measurement_17']),
    ('pt', 'passthrough', ['na_1'])
])

clf_lr = make_pipeline(
    ct, LogisticRegression(solver = 'lbfgs')
)
eval_model('baseline', clf_lr)

,result
name,
baseline,"Valid: 0.58937±0.00380, Train: 0.59190±0.00146"


## Step3: 모델 선택 루틴을 만듭니다.

id,failure

16115, 0.1

16116, 0.2

In [62]:
prd = select_model(clf_lr)
print("자가채점 결과:", roc_auc_score(s_kaggle_ans, prd))

자가채점 결과: 0.5883911870503598


## Step4: 모델 개선을 해봅니다.

In [63]:
# Baseline 튜닝: C
# std: ['loading', 'measurement_1', 'measurement_4', 'measurement_14', 'measurement_17'] + pt: ['na_1'] -> LR

from sklearn.linear_model import LogisticRegression
ct = ColumnTransformer([
    ('std', StandardScaler(), ['loading', 'measurement_1', 'measurement_4', 'measurement_14', 'measurement_17']),
    ('pt', 'passthrough', ['na_1'])
])

clf_lr = make_pipeline(
    ct, LogisticRegression(solver = 'lbfgs', C = 0.03)
)
eval_model('baseline', clf_lr)

,result
name,
baseline,"Valid: 0.58957±0.00417, Train: 0.59194±0.00144"


### LR2

LR + PCA - measurement_0~17 -> STD -> PCA(n_components = 7) + loading -> STD,  PT: na_1, na_2

In [64]:
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA
ct = ColumnTransformer([
    ('std_pca', make_pipeline(StandardScaler(), PCA(n_components = 7)), ['measurement_{}'.format(i) for i in range(0, 18)]),
    ('std', StandardScaler(), ['loading']),
    ('pt', 'passthrough', ['na_1', 'na_2'])
])

clf_lr2 = make_pipeline(
    ct, LogisticRegression(solver = 'lbfgs', C = 0.03)
)
eval_model('clf_lr2', clf_lr2)

,result
name,
baseline,"Valid: 0.58957±0.00417, Train: 0.59194±0.00144"
clf_lr2,"Valid: 0.58901±0.00345, Train: 0.59199±0.00101"


### LR3

Baseline + loading_log

In [65]:
df_train['loading_log'].isna().sum()

0

In [66]:
from sklearn.linear_model import LogisticRegression
ct = ColumnTransformer([
    ('std', StandardScaler(), ['loading_log', 'measurement_1', 'measurement_4', 'measurement_14', 'measurement_17']),
    ('pt', 'passthrough', ['na_1'])
])

clf_lr3 = make_pipeline(
    ct, LogisticRegression(solver = 'lbfgs', C = 0.03)
)
eval_model('clf_lr3', clf_lr3)

,result
name,
baseline,"Valid: 0.58957±0.00417, Train: 0.59194±0.00144"
clf_lr2,"Valid: 0.58901±0.00345, Train: 0.59199±0.00101"
clf_lr3,"Valid: 0.58945±0.00423, Train: 0.59171±0.00147"


In [69]:
from sklearn.preprocessing import FunctionTransformer

ct = ColumnTransformer([
    ('log_std', make_pipeline(FunctionTransformer(np.log, validate=False), StandardScaler()), ['loading']),
    ('std', StandardScaler(), ['measurement_1', 'measurement_4', 'measurement_14', 'measurement_17']),
    ('pt', 'passthrough', ['na_1'])
])

clf_lr3 = make_pipeline(
    ct, LogisticRegression(solver = 'lbfgs', C = 0.03)
)
eval_model('clf_lr3', clf_lr3)

,result
name,
baseline,"Valid: 0.58957±0.00417, Train: 0.59194±0.00144"
clf_lr2,"Valid: 0.58901±0.00345, Train: 0.59199±0.00101"
clf_lr3,"Valid: 0.58945±0.00423, Train: 0.59171±0.00147"


### LDA

LDA + Baseline

In [70]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
ct = ColumnTransformer([
    ('std', StandardScaler(), ['loading', 'measurement_1', 'measurement_4', 'measurement_14', 'measurement_17']),
    ('pt', 'passthrough', ['na_1'])
])

clf_lda = make_pipeline(
    ct, LinearDiscriminantAnalysis()
)
eval_model('clf_lda', clf_lda)

,result
name,
baseline,"Valid: 0.58957±0.00417, Train: 0.59194±0.00144"
clf_lda,"Valid: 0.58963±0.00397, Train: 0.59196±0.00148"
clf_lr2,"Valid: 0.58901±0.00345, Train: 0.59199±0.00101"
clf_lr3,"Valid: 0.58945±0.00423, Train: 0.59171±0.00147"


In [71]:
prd = select_model(clf_lda)
print("자가채점 결과:", roc_auc_score(s_kaggle_ans, prd)) #  0.5883911870503598

자가채점 결과: 0.5892796762589928


### RF

- ['loading', 'na_1', 'na_2'] + ['measurement_{}'.format(i) for i in range(18)]
- RF: RandomForestClassifier: n_estimators=100, max_depth=6, min_samples_split=512, random_state=123
- max_features를 사용해 튜닝

In [72]:
from sklearn.ensemble import RandomForestClassifier

ct = ColumnTransformer([
    ('pt', 'passthrough', ['loading', 'na_1', 'na_2'] + ['measurement_{}'.format(i) for i in range(0, 18)])
])

clf_rf = make_pipeline(
    ct, RandomForestClassifier(n_estimators=100, max_depth=6, min_samples_split=512, random_state=123)
)
eval_model('clf_rf', clf_rf)

,result
name,
baseline,"Valid: 0.58957±0.00417, Train: 0.59194±0.00144"
clf_lda,"Valid: 0.58963±0.00397, Train: 0.59196±0.00148"
clf_lr2,"Valid: 0.58901±0.00345, Train: 0.59199±0.00101"
clf_lr3,"Valid: 0.58945±0.00423, Train: 0.59171±0.00147"
clf_rf,"Valid: 0.58060±0.00466, Train: 0.64370±0.00308"


In [73]:
from sklearn.ensemble import RandomForestClassifier

ct = ColumnTransformer([
    ('pt', 'passthrough', ['loading', 'na_1', 'na_2'] + ['measurement_{}'.format(i) for i in range(0, 18)])
])

clf_rf = make_pipeline(
    ct, RandomForestClassifier(
        n_estimators=100, max_depth=6, min_samples_split=512, random_state=123,
        max_features = 0.75
    )
)
eval_model('clf_rf', clf_rf)

,result
name,
baseline,"Valid: 0.58957±0.00417, Train: 0.59194±0.00144"
clf_lda,"Valid: 0.58963±0.00397, Train: 0.59196±0.00148"
clf_lr2,"Valid: 0.58901±0.00345, Train: 0.59199±0.00101"
clf_lr3,"Valid: 0.58945±0.00423, Train: 0.59171±0.00147"
clf_rf,"Valid: 0.58168±0.00306, Train: 0.63957±0.00235"


### RF2: RandomForestClassifier + LinearDiscriminantAnalysis

In [75]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

ct = ColumnTransformer([
    ('std_lda', make_pipeline(StandardScaler(), LinearDiscriminantAnalysis()), ['measurement_{}'.format(i) for i in range(0, 18)]),
    ('pt', 'passthrough', ['loading', 'na_1', 'na_2'] )
])

clf_rf2 = make_pipeline(
    ct, RandomForestClassifier(
        n_estimators=100, max_depth=6, min_samples_split=512, random_state=123,
        max_features = 0.75
    )
)
eval_model('clf_rf2', clf_rf2)

,result
name,
baseline,"Valid: 0.58957±0.00417, Train: 0.59194±0.00144"
clf_lda,"Valid: 0.58963±0.00397, Train: 0.59196±0.00148"
clf_lr2,"Valid: 0.58901±0.00345, Train: 0.59199±0.00101"
clf_lr3,"Valid: 0.58945±0.00423, Train: 0.59171±0.00147"
clf_rf,"Valid: 0.58168±0.00306, Train: 0.63957±0.00235"
clf_rf2,"Valid: 0.58590±0.00233, Train: 0.62862±0.00279"


### XGB
- ['loading', 'na_1', 'na_2'] + ['measurement_{}'.format(i) for i in range(18)]
- n_estimators=300, learning_rate=0.01, colsample_bytree=0.85
- subsample을 튜닝

In [76]:
import xgboost as xgb

ct = ColumnTransformer([
    ('pt', 'passthrough', ['loading', 'na_1', 'na_2'] + ['measurement_{}'.format(i) for i in range(18)])
])

clf_xgb = make_pipeline(
    ct, 
    xgb.XGBClassifier(n_estimators = 300, max_depth=3, learning_rate=0.01, colsample_bytree=0.85, random_state=123)
)
eval_model('clf_xgb', clf_xgb)

,result
name,
baseline,"Valid: 0.58957±0.00417, Train: 0.59194±0.00144"
clf_lda,"Valid: 0.58963±0.00397, Train: 0.59196±0.00148"
clf_lr2,"Valid: 0.58901±0.00345, Train: 0.59199±0.00101"
clf_lr3,"Valid: 0.58945±0.00423, Train: 0.59171±0.00147"
clf_rf,"Valid: 0.58168±0.00306, Train: 0.63957±0.00235"
clf_rf2,"Valid: 0.58590±0.00233, Train: 0.62862±0.00279"
clf_xgb,"Valid: 0.58269±0.00260, Train: 0.62595±0.00119"


In [77]:
import xgboost as xgb

ct = ColumnTransformer([
    ('pt', 'passthrough', ['loading', 'na_1', 'na_2'] + ['measurement_{}'.format(i) for i in range(18)])
])

# 파라메터 튜닝: subsample - 이상점 / 영향점이 많은 데이터셋에서 조정을 해볼만 합니다.
clf_xgb = make_pipeline(
    ct, 
    xgb.XGBClassifier(n_estimators = 300, max_depth=3, learning_rate=0.01, colsample_bytree=0.85, subsample = 0.75, random_state=123)
)
eval_model('clf_xgb', clf_xgb)

,result
name,
baseline,"Valid: 0.58957±0.00417, Train: 0.59194±0.00144"
clf_lda,"Valid: 0.58963±0.00397, Train: 0.59196±0.00148"
clf_lr2,"Valid: 0.58901±0.00345, Train: 0.59199±0.00101"
clf_lr3,"Valid: 0.58945±0.00423, Train: 0.59171±0.00147"
clf_rf,"Valid: 0.58168±0.00306, Train: 0.63957±0.00235"
clf_rf2,"Valid: 0.58590±0.00233, Train: 0.62862±0.00279"
clf_xgb,"Valid: 0.58327±0.00345, Train: 0.63020±0.00114"


### Voting

- ('baseline', clf_lr), # LR + SFS
- ('lr2', clf_lr2), # LR.2: LR + feature PCA
- ('lda', clf_lda), # LDA
- ('rf2', clf_rf2), # RF + LDA

## Stacking

다른 앙상블 기법인 Stacking을 보여드립니다.


참고용입니다. 지금까지 했던 데이터 처리와 머신러닝 기법을 복습해보기 위해 준비했습니다.